# Actividad 1 · Experimentos
Visión Artificial — MIA UNIR

Notebook limpio para los experimentos. Todas las técnicas y utilidades están en `mejora_imagen.py`; el notebook original (`act1_mejora_imagen.ipynb`) se conserva tal cual.

1. Configuración y carga de las 4 imágenes de Dark Face
2. Comparación múltiple (rejilla + tabla de métricas)
3. Ejemplo de técnicas encadenadas

In [ ]:
# Recarga mejora_imagen.py automáticamente al cambiarlo (sin reiniciar el kernel)
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Todas las funciones (carga, métricas, figuras y técnicas) están en mejora_imagen.py
from mejora_imagen import (
    cargar_imgs_propias, metricas, sigma_ruido, recorte, comparar, Tecnica, Cadena,
    negativo, estiramiento_lineal, transformacion_log, correccion_gamma,
    ecualizacion_global,
    suma_constante, multiplicacion_constante, realce_laplaciano,
)

# Carpeta con las 4 imágenes de Dark Face y carpeta donde se guardan las figuras
DIR_IMGS = Path("imgs")
DIR_FIGS = Path("figures")
DIR_FIGS.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 110

## 1. Carga de imágenes

In [ ]:
imagenes = cargar_imgs_propias(DIR_IMGS, [182, 5069, 5940, 5019])
nombres = list(imagenes)   # ["182", "5069", "5940", "5019"]

pd.DataFrame([
    {"imagen": n, "alto": d["bgr"].shape[0], "ancho": d["bgr"].shape[1],
     "canales": d["bgr"].shape[2], "tipo": str(d["bgr"].dtype)}
    for n, d in imagenes.items()
])

## 2. Comparación múltiple
`comparativa_mult` aplica una lista de `Tecnica` (o `Cadena`) a una imagen y genera la rejilla con histogramas y la tabla de métricas (con `sigma_ruido`). Las etiquetas salen de `str(t)`, así que coinciden siempre con los parámetros aplicados.

Salidas: `figures/{nombre}_mult_{sufijo}.png` y `figures/{nombre}_comparativa_mult_{sufijo}.csv`.

In [ ]:
def comparativa_mult(tecnicas, nombre: str, sufijo: str, titulo: str):
    """Aplica cada Tecnica a la imagen `nombre`: rejilla con histogramas + tabla de métricas.

    Las etiquetas de la figura y de la tabla salen de str(t), así que coinciden
    siempre con el parámetro realmente aplicado.
    """
    I = imagenes[nombre]["gris"]
    
    dic = {"Original": I}
    # sigma_ruido: desviación típica estimada del ruido (Immerkær), además de las métricas habituales
    filas = [{"imagen": f"{nombre}", "técnica": "Original",  **metricas(I), "sigma_ruido": sigma_ruido(I)}]
    for t in tecnicas:
        I_t = t(I)
        dic[str(t)] = I_t
        filas.append({"imagen": f"{nombre}", "técnica": str(t),
                      **metricas(I_t), "sigma_ruido": sigma_ruido(I_t)})
    
    
    comparar(dic,
             titulo=f"{nombre} · {titulo}", guardar=DIR_FIGS / f"{nombre}_mult_{sufijo}.png", columnas=3, filas=2)
    
    tabla_comparativa = pd.DataFrame(filas).set_index(["imagen","técnica"]).round(3)
    tabla_comparativa.to_csv(DIR_FIGS / f"{nombre}_comparativa_mult_{sufijo}.csv")
    print(tabla_comparativa)


def gamma_comparativa_mult(gammas, nombre: str):
    tecnicas = [Tecnica(correccion_gamma, "Gamma", {"γ": g}) for g in gammas]
    comparativa_mult(tecnicas, nombre, sufijo="gamma", titulo="corrección gamma")

In [ ]:
def multiplicacion_comparativa_mult(ks, nombre: str):
    tecnicas = [Tecnica(multiplicacion_constante, "Multiplicación", {"k": k}) for k in ks]
    comparativa_mult(tecnicas, nombre, sufijo="multiplicacion", titulo="multiplicación por constante")

## 3. Ejemplo de técnicas encadenadas
Una `Cadena` es la composición T = T₂ ∘ T₁ (primero T₁, luego T₂) y se construye con `>>`. Se usa igual que una `Tecnica`: `c(I)` la aplica y `str(c)` da la etiqueta, así que entra directamente en `comparativa_mult`.

Ejemplo: gamma y ecualización global por separado y encadenadas en los dos órdenes, para ver que en general T₂ ∘ T₁ ≠ T₁ ∘ T₂. Entre pasos se redondea y recorta a [0, 255] (cada técnica devuelve `uint8`). Los parámetros son de ejemplo.

In [ ]:
g  = Tecnica(correccion_gamma, "Gamma", {"γ": 0.6})
eq = Tecnica(ecualizacion_global, "Ecualización global")

g_eq = g >> eq   # primero gamma, luego ecualización
eq_g = eq >> g   # primero ecualización, luego gamma

# Comprobación: la cadena equivale a las llamadas anidadas
I = imagenes[nombres[0]]["gris"]
print(str(g_eq), "≡ llamadas anidadas:",
      np.array_equal(g_eq(I), ecualizacion_global(correccion_gamma(I, 0.6))))
print(str(eq_g))

In [ ]:
comparativa_mult([g, eq, g_eq, eq_g], nombres[0],
                 sufijo="cadena_ejemplo", titulo="técnicas encadenadas (ejemplo)")

In [ ]:
for nombre in nombres:
    multiplicacion_comparativa_mult(np.linspace(2,8,7).tolist(), nombre)

## Logaritmo

In [ ]:
def comparativa(tecnicas, nombre: str, sufijo: str, titulo: str, **kw_comparar):
    """Rejilla «Original + cada técnica» sobre la imagen `nombre` (grises), con histogramas.

    tecnicas     lista de Tecnica o Cadena; la etiqueta de cada panel es str(t),
                 así que coincide con los parámetros realmente aplicados.
    sufijo       nombre del PNG: figures/{nombre}_{sufijo}.png (distinto por experimento
                 para no sobrescribir otras figuras).
    kw_comparar  se pasan tal cual a comparar (filas, columnas, hist_log, tam_letra...).
    """
    I = imagenes[nombre]["gris"]
    # Cada Tecnica es una función: hay que aplicarla a I para tener la imagen de salida
    salidas = {"Original": I, **{str(t): t(I) for t in tecnicas}}
    comparar(salidas, titulo=f"{nombre} · {titulo}",
             guardar=DIR_FIGS / f"{nombre}_{sufijo}.png", **kw_comparar)

In [ ]:
tecnicas_log = [Tecnica(transformacion_log, "Log")]   # añade aquí las que quieras comparar

for nombre in nombres:
    comparativa(tecnicas_log, nombre, sufijo="log", titulo="transformación logarítmica")

## Estiramiento lineal

In [ ]:
# Percentiles 1–99 (valores por defecto); se ponen explícitos para que salgan en la etiqueta
tecnicas_estiramiento = [Tecnica(estiramiento_lineal, "Estiramiento lineal", {"p_inf": 1, "p_sup": 99})]

for nombre in nombres:
    comparativa(tecnicas_estiramiento, nombre, sufijo="estiramiento_lineal", titulo="estiramiento lineal")

## Ecualización Global histograma

In [ ]:
tecnicas_ecualizacion = [Tecnica(ecualizacion_global, "Ecualización global")]

for nombre in nombres:
    comparativa(tecnicas_ecualizacion, nombre, sufijo="ecualizacion_global", titulo="ecualización global del histograma")